# Supplementary Fig. 2g: first 4 seconds of simulated dense probe traces, spontaneous regime, one site per layer

**author**: laquitainesteeve@gmail.com

**purpose**: reproduce panel g of Supplementary Fig. 2: first 4 seconds of the raw (unfiltered) traces of the first recording site of each cortical layer of the simulation of the spontaneous regime with the Horvath dense probe (`dataset/00_raw/recording_dense_probe1`, `probe2` and `probe3`, 20 kHz). Source data and svg are saved in figures/8_source_data/supp_fig2g/.

**method (as in the paper's notebook supp_fig2_AE_validation_sws.ipynb)**: the site is the first one (in channel order) of the layer (layers 2 and 3 merged); layers 1 and 2/3 are read from probe 1, layers 4 and 5 from probe 2 and layer 6 from probe 3, the depth of the dense probe that covers each layer; the y-axis spans the minimum and maximum of the 4 seconds.

In [1]:
# SETUP PACKAGES
import os
import numpy as np
import pandas as pd
import warnings
from matplotlib import pyplot as plt
import spikeinterface as si
warnings.simplefilter(action="ignore")

# SET PATHS
PROJ_PATH = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(PROJ_PATH)
RECORDING_PATHS = {"L1": "dataset/00_raw/recording_dense_probe1", "L2_3": "dataset/00_raw/recording_dense_probe1", "L4": "dataset/00_raw/recording_dense_probe2",
                   "L5": "dataset/00_raw/recording_dense_probe2", "L6": "dataset/00_raw/recording_dense_probe3"}

# PARAMETERS
SFREQ = 20000                   # sampling frequency (Hz)
DUR_SEC = 4                     # duration (seconds)
LAYERS = ["L1", "L2_3", "L4", "L5", "L6"]
LAYER_LABELS = ["L1", "L2/3", "L4", "L5", "L6"]
LAYER_COLORS = [(0.7137, 0.6157, 0.3765), (0.7608, 0.4549, 0.5843), (0.6431, 0.5333, 0.6627), (0.3294, 0.4274, 0.4980), (0.5647, 0.7294, 0.5098)]
TRACE_COLOR = (0.3, 0.3, 0.3)

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 0.3
plt.rcParams["axes.linewidth"] = 1
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["xtick.major.width"] = 1
plt.rcParams["ytick.major.width"] = 1
plt.rcParams["xtick.major.size"] = 3.5
plt.rcParams["ytick.major.size"] = 3.5
savefig_cfg = {"transparent": True, "dpi": 300}

In [ ]:
# Nature figure guidelines for every saved figure: Arial, 5-7 pt, editable text, images of at least 300 dpi, a .pdf next to each .svg
import os, sys
sys.path.insert(0, os.getcwd())     # the repository (the notebook moved there above)
from src.nodes import figure_style
figure_style.apply()

## Load the first 4 seconds of the first site of each layer

In [2]:
%%time

df = pd.DataFrame({"time_s": np.arange(DUR_SEC * SFREQ) / SFREQ})
site_ids = {}
for layer in LAYERS:
    Rec = si.load_extractor(RECORDING_PATHS[layer]).frame_slice(0, DUR_SEC * SFREQ)       # lazy: only 4 seconds are read
    layer_of_site = np.array(["L2_3" if x in ("L2", "L3") else x for x in Rec.get_property("layers")])
    site_id = Rec.channel_ids[layer_of_site == layer][0]      # first site of the layer
    site_ids[layer] = site_id
    df[f"{layer}_voltage_uV"] = Rec.get_traces(channel_ids=[site_id])[:, 0]
print("sites:", site_ids)
display(df.describe().loc[["min", "max"]].round(1))

sites: {'L1': '15', 'L2_3': '0', 'L4': '22', 'L5': '0', 'L6': '13'}


,time_s,L1_voltage_uV,L2_3_voltage_uV,L4_voltage_uV,L5_voltage_uV,L6_voltage_uV
min,0.0,-191.5,-175.2,-308.3,-275.9,-455.3
max,4.0,169.0,191.2,198.1,195.0,266.1


CPU times: user 28.5 ms, sys: 12.4 ms, total: 41 ms
Wall time: 79 ms


## Plot, same as the paper's panel c

In [3]:
# paper geometry (paper/figures/supp_fig2.pdf): canvas 141.4 x 151.6 pt, Arial 5 pt text, 7 pt title, editable (src/nodes/validation/trace_plots.py)
from src.nodes.validation.trace_plots import plot_traces, SIM_DENSE_SPONT

def plot_source_data(df: pd.DataFrame):
    return plot_traces(df, SIM_DENSE_SPONT)

## Plot and save the figure and the source data

In [4]:
%%time

fig = plot_source_data(df)
os.makedirs("figures/8_source_data/supp_fig2g", exist_ok=True)
plt.savefig("figures/8_source_data/supp_fig2g/supp_fig2g.svg", **savefig_cfg)
df.to_excel("figures/8_source_data/supp_fig2g/supp_fig2g.xlsx", index=False)

## Check: replot from the saved source data

In [5]:
%%time

df = pd.read_excel("figures/8_source_data/supp_fig2g/supp_fig2g.xlsx")
fig = plot_source_data(df)